# Artificial Intelligence — Lab 1
## Intelligent Agents and Problem Formulation

**Course Learning Outcome — CLO1**  
Describe the fundamentals of Artificial Intelligence, including intelligent agents, environments, state spaces, actions, goals, and problem formulation.

**Environment:** Python 3 / Jupyter Notebook  
**Submission:** completed notebook with required code, predictions, traces, justifications, outputs, and reflection answers.

> **Assessment principle:** Working code is only part of the evidence. Most marks come from your ability to **predict, justify, trace, modify, and explain** the AI model you implement.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. Agent & PEAS reasoning | 15 min | Describe the task environment and agent |
| 2. Reflex agent | 25 min | Predict, implement, and trace condition–action rules |
| 3. Problem formulation | 35 min | Implement states, actions, transitions, goal, and cost |
| 4. Path-cost reasoning | 25 min | Distinguish states, paths, and weighted costs |
| 5. Debugging, reflection & viva | 20 min | Diagnose errors and defend modeling choices |
| **Total** | **120 min** | |

> **Main idea:** A useful AI implementation begins with a precise model of the **agent**, its **environment**, and the **problem** it is trying to solve.

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish an **agent**, **environment**, **percept**, and **action**;
2. describe an agent task using **PEAS**;
3. predict the behavior of a simple agent before executing code;
4. implement and justify a simple **agent function**;
5. formulate a problem using $P=(S,A,s_0,T,G,c)$;
6. connect each mathematical component of a problem formulation to its Python representation;
7. distinguish a **state**, **path**, **state space**, and **search tree**;
8. diagnose a small modeling/programming error and explain the correction;
9. analyze how a change in the formulation affects reachable states and path cost.

## Before You Start

Use only the Python standard library in this lab. The code is intentionally lightweight so that the focus remains on **AI reasoning**, not package installation.

### About AI tools
Follow your instructor's and institution's policy on generative AI. If AI assistance is permitted, it may help you explore syntax or ideas, but **generated code is not evidence of understanding by itself**.

You may be asked to:
- predict an output before running the code;
- explain a selected line or function;
- justify a modeling decision;
- correct a deliberately faulty implementation;
- modify your solution during the lab.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple, Set

print("Lab 1 environment ready.")

# Part I — From an AI Task to an Agent Description

A PEAS description identifies **Performance measure, Environment, Actuators, and Sensors**.

## Task 1.1 — PEAS and Environment Classification

For the two-room robotic vacuum, complete PEAS and classify the environment as fully/partially observable, deterministic/stochastic, episodic/sequential, static/dynamic, discrete/continuous, and single-/multi-agent. Justify each choice briefly.

Finally, explain what changes if the sensor reports dirt status but not the current room.

**Your answer / interpretation:**

# Part II — Predict, Implement, and Explain a Simple Reflex Agent

We model a small **two-room vacuum world**.

A world state is represented as:

```text
(location, room_A_status, room_B_status)
```

For example,

```text
('A', 'Dirty', 'Clean')
```

means that the agent is in room A, room A is dirty, and room B is clean.

In [ ]:
@dataclass
class VacuumWorld:
    location: str = "A"
    room_a: str = "Dirty"
    room_b: str = "Dirty"

    def percept(self) -> Tuple[str, str]:
        """Return (current_location, status_of_current_room)."""
        status = self.room_a if self.location == "A" else self.room_b
        return self.location, status

    def execute(self, action: str) -> None:
        """Apply one action to the environment."""
        if action == "Suck":
            if self.location == "A":
                self.room_a = "Clean"
            else:
                self.room_b = "Clean"
        elif action == "Right" and self.location == "A":
            self.location = "B"
        elif action == "Left" and self.location == "B":
            self.location = "A"
        elif action == "NoOp":
            pass
        else:
            raise ValueError(f"Invalid action {action!r} for location {self.location!r}")

    def is_clean(self) -> bool:
        return self.room_a == "Clean" and self.room_b == "Clean"

    def state(self) -> Tuple[str, str, str]:
        return self.location, self.room_a, self.room_b

## Task 2.1 — Predict the Reflex Rules

Predict the action for each percept before coding:

| Percept | Predicted action | Why? |
|---|---|---|
| `("A","Dirty")` | | |
| `("A","Clean")` | | |
| `("B","Dirty")` | | |
| `("B","Clean")` | | |

**Your prediction / justification:**

## Task 2.2 — Complete the Agent Function

Implement the three rules above.

Your function must return exactly one of `"Suck"`, `"Left"`, or `"Right"`.

In [ ]:
def simple_reflex_agent(percept: Tuple[str, str]) -> str:
    location, status = percept

    # TODO: implement the three condition-action rules.
    raise NotImplementedError("Complete simple_reflex_agent().")

assert simple_reflex_agent(("A", "Dirty")) == "Suck"
assert simple_reflex_agent(("B", "Dirty")) == "Suck"
assert simple_reflex_agent(("A", "Clean")) == "Right"
assert simple_reflex_agent(("B", "Clean")) == "Left"
print("Agent-function tests passed.")

def run_vacuum_episode(world: VacuumWorld, max_steps: int = 8) -> List[dict]:
    history = []

    for step in range(max_steps):
        percept = world.percept()
        action = simple_reflex_agent(percept)
        before = world.state()
        world.execute(action)
        after = world.state()

        history.append({
            "step": step + 1,
            "percept": percept,
            "action": action,
            "state_before": before,
            "state_after": after,
            "all_clean": world.is_clean(),
        })

        if world.is_clean():
            break

    return history


def print_episode(history: List[dict]) -> None:
    for row in history:
        print(
            f"Step {row['step']}: percept={row['percept']}  "
            f"action={row['action']:<5}  "
            f"state={row['state_after']}"
        )

## Task 2.3 — Trace and Explain One Episode

For initial state `("A","Dirty","Dirty")`, predict the first three percept–action–state transitions. Run the episode and compare the actual trace with your prediction. Explain why the reflex agent needs the current location in its percept.

**Your trace / reasoning:**

In [ ]:
world = VacuumWorld(location="A", room_a="Dirty", room_b="Dirty")
history = run_vacuum_episode(world)
print_episode(history)

# Part III — Problem Formulation

A search problem can be represented by

$$P=(S,A,s_0,T,G,c).$$

We now formulate the grid-navigation problem without implementing a search strategy.

In [ ]:
GRID = [
    ".......",
    ".###...",
    "...#...",
    "...#.#.",
    ".......",
]

START = (4, 0)
GOAL = (0, 6)


def show_grid(grid, start=None, goal=None, current=None):
    canvas = [list(row) for row in grid]
    if start is not None:
        r, c = start
        canvas[r][c] = "S"
    if goal is not None:
        r, c = goal
        canvas[r][c] = "G"
    if current is not None and current not in {start, goal}:
        r, c = current
        canvas[r][c] = "A"
    print("\n".join(" ".join(row) for row in canvas))

show_grid(GRID, START, GOAL)

## Task 3.1 — Legal Actions and Formal Components

Predict the legal actions from `(4,0)`, `(0,0)`, `(2,2)`, and `(0,6)`.

Then identify what $S$, $A$, $s_0$, $T$, $G$, and $c$ mean in this grid. Explain the difference between a **state** and a **path**.

**Your prediction / justification:**

## Task 3.2 — Complete the Grid Problem Class

Complete `actions`, `result`, and `goal_test`.

### Required behavior

- `actions(state)` returns only actions that lead to valid, non-obstacle cells.
- `result(state, action)` returns the successor state produced by the action.
- `goal_test(state)` returns `True` exactly when `state == self.goal`.

In [ ]:
@dataclass
class GridProblem:
    grid: List[str]
    initial: Tuple[int, int]
    goal: Tuple[int, int]

    DELTAS = {
        "Up": (-1, 0),
        "Down": (1, 0),
        "Left": (0, -1),
        "Right": (0, 1),
    }

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        r, c = state
        return 0 <= r < len(self.grid) and 0 <= c < len(self.grid[0])

    def passable(self, state: Tuple[int, int]) -> bool:
        r, c = state
        return self.grid[r][c] != "#"

    def actions(self, state: Tuple[int, int]) -> List[str]:
        # TODO: return all legal actions from state.
        raise NotImplementedError

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        # TODO: return the next state.
        # Raise ValueError if action is not legal from this state.
        raise NotImplementedError

    def goal_test(self, state: Tuple[int, int]) -> bool:
        # TODO: return True iff state is the goal.
        raise NotImplementedError

    def step_cost(self, state, action, next_state) -> float:
        return 1.0

    def successors(self, state: Tuple[int, int]):
        return [
            (action, self.result(state, action), self.step_cost(state, action, self.result(state, action)))
            for action in self.actions(state)
        ]

problem = GridProblem(GRID, START, GOAL)

assert problem.goal_test(GOAL) is True
assert problem.goal_test(START) is False
assert set(problem.actions((4, 0))) == {"Up", "Right"}
assert problem.result((4, 0), "Up") == (3, 0)
assert problem.result((4, 0), "Right") == (4, 1)
assert "Up" not in problem.actions((0, 0))

print("Grid formulation tests passed.")

## Task 3.3 — Map the Code and Debug the Goal Test

Explain where `actions`, `result`, `goal_test`, and `step_cost` correspond to the mathematical formulation.

Then diagnose:

```python
def goal_test(self, state):
    return state[0] == self.goal[0]
```

Give a false goal state and write the correct condition.

**Your answer / interpretation:**

# Part IV — Path Cost and a Small Extension

So far every move costs 1. In many AI problems, two paths with the same number of actions can have different total costs.

Suppose some cells represent difficult terrain:

- normal cell: cost 1;
- rough cell: cost 3;
- very rough cell: cost 5.

The total path cost is

$$
g(n)=\sum_{i=1}^{k} c(s_{i-1},a_i,s_i).
$$

This distinction becomes important when we study **Uniform-Cost Search** and **A\***.

In [ ]:
def path_cost(problem: GridProblem, states: List[Tuple[int, int]]) -> float:
    """Compute the cost of a path represented as a list of consecutive states."""
    if len(states) < 2:
        return 0.0

    total = 0.0
    for current, nxt in zip(states, states[1:]):
        legal_action = None
        for action in problem.actions(current):
            if problem.result(current, action) == nxt:
                legal_action = action
                break
        if legal_action is None:
            raise ValueError(f"{current} -> {nxt} is not a legal transition")
        total += problem.step_cost(current, legal_action, nxt)
    return total

## Task 4.1 — Weighted Grid Challenge

Create a subclass `WeightedGridProblem` that overrides `step_cost`.

Use:

```python
terrain_cost = {
    (4, 2): 3,
    (4, 3): 3,
    (0, 4): 5,
}
```

The cost of entering a cell should equal the value in the dictionary, or 1 if the cell is not listed.

In [ ]:
class WeightedGridProblem(GridProblem):
    def __init__(self, grid, initial, goal, terrain_cost):
        super().__init__(grid, initial, goal)
        self.terrain_cost = terrain_cost

    def step_cost(self, state, action, next_state) -> float:
        # TODO: return the cost of entering next_state.
        raise NotImplementedError


terrain_cost = {
    (4, 2): 3,
    (4, 3): 3,
    (0, 4): 5,
}

# weighted_problem = WeightedGridProblem(GRID, START, GOAL, terrain_cost)

## Task 4.2 — Interpret Weighted Path Cost

Explain why changing terrain costs changes $c$ but not necessarily $S$. Why can the path with fewer actions cost more? Which later search strategy will optimize cumulative cost $g(n)$?

**Your answer / interpretation:**

## Individual Understanding Check

Your instructor may ask one short question about your notebook:

- What is PEAS and why is the performance measure important?
- What is the difference between a state and a path?
- Show where one component of $P=(S,A,s_0,T,G,c)$ appears in your code.

## Reflection

Answer concisely:

1. Which result or trace best illustrates the algorithm's behavior?
2. Which implementation or modeling mistake would most seriously change the conclusion?

**Your reflection:**

## Submission & Assessment

Before submitting, verify that all required TODO code, predictions/traces, experimental results, interpretations, debugging answers, and reflection answers are complete.

| Component | Marks |
|---|---:|
| Correct implementation | **2** |
| Algorithmic / modeling justification | **3** |
| Experimental analysis | **2** |
| Trace / prediction / debugging | **1** |
| Individual understanding check | **1** |
| Code quality / submission completeness | **1** |
| **Total** | **10** |

> Correct code without adequate explanation earns only a limited portion of the marks.